# Liu2024 — Multi-window Riemannian ensemble x frozen S-JEPA calibrated stacking

**What this tests.** Two branches already individually shown to beat chance under an *honest*
(nested-CV, no test-set peeking) evaluation on this dataset:

- **Branch A — Riemannian geometry**, computed here as an **ensemble average across several fixed
  time windows** (not a single inner-CV-*selected* best window). The existing hybrid notebook
  (`liu2024_sjepa_twfb_hybrid.ipynb`) picks one winning window via inner-CV argmax over only 3
  candidates with 3-fold inner CV on ~40 trials/subject — a high-variance selection given the
  sample size. Averaging predicted probabilities across a small fixed window grid is a standard
  variance-reduction move (same spirit as filter-bank ensembling) and needs **no selection step at
  all**, so it cannot leak by construction.
- **Branch B — frozen S-JEPA embeddings** (`liu2024_sjepa_embeddings_lda.ipynb`), deterministic and
  label-free per trial — also leak-free by construction.

**Fusion — calibrated stacking, not feature concatenation.** Per outer fold: an **inner CV on the
train split only** produces out-of-fold probabilities for both branches; a logistic-regression
meta-learner is fit on those out-of-fold probabilities (standard stacking, avoids the meta-learner
overfitting to in-sample base-learner predictions); base learners are then refit on the *full*
outer-train split and applied once to the outer-test split, together with the meta-learner. This
is a materially lower-variance, more rigorously honest version of the existing hybrid notebook's
raw-concatenation + single-LDA fusion.

**Why this is scientifically reasonable.** Both branches are already individually significant vs.
chance under honest evaluation elsewhere in this codebase, and neither branch nor the fusion step
here has any test-set-visible tuning knob: window averaging is fixed (not selected), S-JEPA
features are frozen, and the only fitted meta-learner is trained on **inner**-CV out-of-fold
probabilities, never on the outer-test fold.

**STOP / historical notebook.** A Full50 prototype already reached 52.13% Riemann, 54.25% frozen
S-JEPA, and 50.25% stacking; later multiscale and same-fold fusion experiments were also null.
Do not run or scale this notebook without a materially new prespecified hypothesis. See the
workspace-root `AGENTS.md` section 2e.

**Subject sets are always reported separately** — full 50-subject vs. the Lv et al. 2025 14-subject
subset (`lv14_subject_ids` in CONFIG) — never mixed in one row of a results table.


# 1. Setup

In [14]:
raise RuntimeError('CLOSED multiwindow stacking experiment: see AGENTS.md section 2e.')
import os, sys, glob, json, time, hashlib, builtins, platform, random
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import scipy.io as sio
from scipy.signal import butter, iirnotch, lfilter
from scipy.stats import wilcoxon

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix

import mne; mne.set_log_level("ERROR")
import torch
from pyriemann.tangentspace import TangentSpace

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as e:
    HAVE_MPL = False; print("matplotlib unavailable:", e)

from braindecode.models import SignalJEPA_PreLocal

print("python", platform.python_version(), "| numpy", np.__version__, "| torch", torch.__version__)


[2026-07-07 09:12:25] python 3.11.15 | numpy 2.4.3 | torch 2.10.0


# 2. Configuration *(edit this one cell, or apply a sweep JSON, then re-run)*

In [15]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    "experiment_name": "multiwindow_riemann_sjepa_stacking",
    "config_note": ("Honest nested-CV: multi-window Riemannian probability ensemble (no "
                     "test-visible window selection) fused with frozen S-JEPA embeddings via "
                     "inner-CV-fit logistic stacking. HISTORICAL STOPPED CONFIG -- do not "
                     "scale without a new prespecified hypothesis; see AGENTS.md section 2e."),

    "source_roots": [
        str(WORKING_DIR.parent / "Liu2024_matlab_code" / "sourcedata"),
        str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    ],
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-multiwindow-riemann-sjepa-stacking"),

    # ---- subjects -----------------------------------------------------------------
    # SMOKE TEST: 2 subjects only. Reference list for the Lv et al. 2025 14-subject
    # subset (see CLAUDE.md sec.3 / memory:lv-liu-subject-mapping) is kept here so full
    # runs can target it explicitly. Full-50 and Lv-14 results must be reported as
    # SEPARATE rows/tables -- never averaged together.
    "subjects_to_use": None,
    "lv14_subject_ids": [1, 3, 7, 9, 10, 11, 14, 15, 17, 29, 31, 32, 37, 41],

    # ---- channels / onset (faithful to the paper-faithful + hybrid notebooks) -----
    "channel_indices": list(range(0, 17)) + list(range(18, 30)),  # 29 ch, drop CPz (idx 17)
    "marker_channel_index": 32, "onset_marker_value": 2,
    "onset_plausible_range": [800, 1300], "onset_fallback_sample": 1003,
    "native_sfreq": 500, "preroll_samples": 800, "mi_segment_samples": 2000,  # 0-4s @ 500Hz

    # ---- Branch A: multi-window Riemannian ensemble --------------------------------
    # SMOKE TEST: 2 windows x 2 bands = 4 (window, band) tangent blocks.
    # FULL RUN: 3 windows x 8 bands -- matches liu2024_sjepa_twfb_hybrid.ipynb's grid
    # exactly, so fusion vs. that notebook's single-window-selection result is a fair
    # apples-to-apples comparison of "ensemble" vs. "select" as the only changed variable.
    "tw_starts_s": [0.0, 2.0],
    "tw_len_s": 2.0,
    "freq_bands": [(8, 30), (8, 15)],
    "notch_freq": 50.0, "notch_Q": 6.0, "butter_order": 2,
    "cov_shrinkage": 0.10, "tangent_metric": "logeuclid",

    # ---- Branch B: frozen S-JEPA embeddings -----------------------------------------
    "sjepa_repo_id": "braindecode/signal-jepa_without-chans",
    "sjepa_checkpoint_path": None,
    "sjepa_sfreq": 128, "sjepa_window_start_s": 1.5, "sjepa_window_samples": 537,
    "sjepa_bandpass_hz": (0.5, 40.0),
    "embedding_hook": "feature_encoder", "embedding_pool": "mean",
    # Read from this cache dir if per-subject .npz files already exist (written by
    # liu2024_sjepa_embeddings_lda.ipynb's frozen-embedding-cache cell); otherwise
    # computed inline. No cache exists on this Mac yet, so the smoke test computes inline.
    "sjepa_embedding_cache_dir": str(WORKING_DIR / "artifacts" / "liu2024_sjepa_embeddings_lda" / "embeddings"),

    # ---- fusion / stacking -----------------------------------------------------------
    "meta_C": 1.0,               # inverse regularization strength for the logistic meta-learner

    # ---- honest nested-CV evaluation --------------------------------------------------
    "n_repeats": 1,       # SMOKE TEST. Full run: 10 (matches liu2024_sjepa_twfb_hybrid.ipynb).
    "test_size": 0.40,
    "inner_folds": 2,     # SMOKE TEST. Full run: 3.
    "random_state": 2026,

    "device": "auto",
}

BANDS = [tuple(b) for b in CONFIG["freq_bands"]]
FS = CONFIG["native_sfreq"]
CH = CONFIG["channel_indices"]
NCH = len(CH)
DEVICE = ("cuda" if torch.cuda.is_available() else "cpu") if CONFIG["device"] == "auto" else CONFIG["device"]
np.random.seed(CONFIG["random_state"]); random.seed(CONFIG["random_state"]); torch.manual_seed(CONFIG["random_state"])
print(f"device={DEVICE}  channels={NCH}  bands={len(BANDS)}  windows={CONFIG['tw_starts_s']}  "
      f"subjects={CONFIG['subjects_to_use']}  n_repeats={CONFIG['n_repeats']}  inner_folds={CONFIG['inner_folds']}")


[2026-07-07 09:12:25] device=cpu  channels=29  bands=2  windows=[0.0, 2.0]  subjects=None  n_repeats=1  inner_folds=2


## 2.1 Logging, Run ID, artifact dir

In [16]:
def create_run_id():
    h = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{datetime.now().strftime('%Y%m%d_%H%M')}_{h}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")
def _w(stream, t):
    try: stream.write(t)
    except UnicodeEncodeError:
        enc = getattr(stream, "encoding", None) or "utf-8"; stream.write(t.encode(enc, "replace").decode(enc, "replace"))
def _tprint(*a, **k):
    sep = k.pop("sep", " "); end = k.pop("end", "\n"); k.pop("flush", False); k.pop("file", None)
    msg = sep.join(str(x) for x in a); lead = len(msg) - len(msg.lstrip("\n")); body = msg[lead:]
    if lead: _w(sys.stdout, "\n"*lead); _w(_LOG, "\n"*lead)
    if body:
        s = f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {body}"; _w(sys.stdout, s+end); _w(_LOG, s+end)
    else: _w(sys.stdout, end); _w(_LOG, end)
builtins.print = _tprint

with open(ARTIFACT_DIR/"config.json", "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)

print("="*70)
print(f"Experiment: {CONFIG['experiment_name']}")
print(f"Note:       {CONFIG['config_note']}")
print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print("="*70)


[2026-07-07 09:12:25] ======================================================================
[2026-07-07 09:12:25] Experiment: multiwindow_riemann_sjepa_stacking
[2026-07-07 09:12:25] Note:       Historical saved output from the stopped multi-window stacking prototype.
[2026-07-07 09:12:25] Run ID:     20260707_0912_700a2c85
[2026-07-07 09:12:25] Artifacts:  /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-multiwindow-riemann-sjepa-stacking/20260707_0912_700a2c85
[2026-07-07 09:12:25] ======================================================================


# 3. Data loading + MI onset

Same onset-detection logic as `liu2024_twfb_dgfmdm_paper_faithful.ipynb` (marker==2 scan, plausible-range check, per-subject median-onset fallback for out-of-range trials) -- more robust than a fixed-cue assumption given real onset jitter in this dataset.

In [17]:
def find_files(roots):
    for r in roots:
        f = sorted(glob.glob(os.path.join(r, "sub-*", "sub-*_eeg.mat")))
        if f:
            print("source:", r, f"({len(f)} subjects)")
            return f
    raise FileNotFoundError(roots)

def load_raw(path):
    m = sio.loadmat(path); eeg = m["eeg"][0, 0]
    raw = np.asarray(eeg["rawdata"], dtype=np.float64)
    lab = np.asarray(eeg["label"]).ravel().astype(int)
    if set(np.unique(lab)).issubset({1, 2}): lab = lab - 1
    mark = raw[:, CONFIG["marker_channel_index"], :]
    lo, hi = CONFIG["onset_plausible_range"]
    first = []
    for t in range(raw.shape[0]):
        idx = np.where(mark[t] == CONFIG["onset_marker_value"])[0]
        in_range = idx[(idx >= lo) & (idx <= hi)]
        first.append(int(in_range[0]) if in_range.size else (int(idx[0]) if idx.size else -1))
    plausible = [o for o in first if lo <= o <= hi]
    med = int(np.median(plausible)) if plausible else CONFIG["onset_fallback_sample"]
    onsets = [o if lo <= o <= hi else med for o in first]
    return raw, lab, np.asarray(onsets, dtype=int)

print("data loader defined")


[2026-07-07 09:12:25] data loader defined


# 4. Branch A -- multi-window Riemannian probability ensemble

For each candidate window start in `tw_starts_s`: per-band tangent-space features (`TangentSpace` fit on **train only**) concatenated across bands, then a shrinkage-LDA base learner (fit on **train only**) producing calibrated class probabilities on the query set. The final Branch-A probability is the **plain average across windows** -- a fixed ensemble, never a test-visible *selection* among windows.

In [18]:
_wo = CONFIG["notch_freq"]/(FS/2)
NB, NA = iirnotch(_wo, _wo/CONFIG["notch_Q"])
BBA = {bd: butter(CONFIG["butter_order"], [bd[0]/(FS/2), bd[1]/(FS/2)], btype="band") for bd in BANDS}

def band_segment(raw, onsets, bd):
    b, a = BBA[bd]; pre = CONFIG["preroll_samples"]; L = CONFIG["mi_segment_samples"]; n = raw.shape[0]
    out = np.zeros((n, NCH, L))
    for t in range(n):
        s0 = onsets[t] - pre
        seg = raw[t][:, s0:s0+pre+L][CH, :]
        seg = lfilter(NB, NA, seg, axis=1); seg = lfilter(b, a, seg, axis=1)
        out[t] = seg[:, pre:pre+L]
    return out

def covs_window(seg, w_start_s, w_len_s):
    s = int(round(w_start_s*FS)); e = s + int(round(w_len_s*FS)); n = seg.shape[0]
    C = np.zeros((n, NCH, NCH)); g = CONFIG["cov_shrinkage"]
    for t in range(n):
        X = seg[t][:, s:e]; c = X @ X.T; c = c/np.trace(c); c = (1-g)*c + g*np.eye(NCH)/NCH
        C[t] = c
    return C

def tangent_block(cov_by_band, tr, te):
    Ftr, Fte = [], []
    for bd in BANDS:
        ts = TangentSpace(metric=CONFIG["tangent_metric"]); ts.fit(cov_by_band[bd][tr])
        Ftr.append(ts.transform(cov_by_band[bd][tr])); Fte.append(ts.transform(cov_by_band[bd][te]))
    return np.concatenate(Ftr, 1), np.concatenate(Fte, 1)

def make_lda():
    return LinearDiscriminantAnalysis(solver="lsqr", shrinkage="auto")

def riemann_window_proba(seg_by_band, y, tr, te, w_start_s, w_len_s):
    covb = {bd: covs_window(seg_by_band[bd], w_start_s, w_len_s) for bd in BANDS}
    Ftr, Fte = tangent_block(covb, tr, te)
    sc = StandardScaler().fit(Ftr)
    clf = make_lda().fit(sc.transform(Ftr), y[tr])
    return clf.predict_proba(sc.transform(Fte))

def riemann_multiwindow_proba(seg_by_band, y, tr, te):
    probs = [riemann_window_proba(seg_by_band, y, tr, te, s, CONFIG["tw_len_s"]) for s in CONFIG["tw_starts_s"]]
    return np.mean(probs, axis=0)

print("Branch A (multi-window Riemannian ensemble) defined")


[2026-07-07 09:12:25] Branch A (multi-window Riemannian ensemble) defined


# 5. Branch B -- frozen S-JEPA embeddings

Same forward-hook extraction as `liu2024_sjepa_embeddings_lda.ipynb` / `liu2024_sjepa_twfb_hybrid.ipynb`: a **frozen, non-fine-tuned** `SignalJEPA_PreLocal` encoder, mean-pooled `feature_encoder` output per trial. Deterministic and label-free by construction, so there is no leakage risk to manage in this branch -- only the downstream LDA base learner is fit per fold (train only).

In [19]:
EEG_NAMES = ["Fp1","Fp2","Fz","F3","F4","F7","F8","FCz","FC3","FC4","FT7","FT8","Cz","C3","C4","T3","T4",
             "CP3","CP4","TP7","TP8","Pz","P3","P4","T5","T6","Oz","O1","O2"]   # 29, CPz dropped

def sjepa_windows(raw, onsets):
    sf = CONFIG["sjepa_sfreq"]; W = CONFIG["sjepa_window_samples"]
    eeg = raw[:, CH, :] * 1e-6
    n_tr, _, n_t = eeg.shape
    cont = eeg.transpose(1, 0, 2).reshape(NCH, n_tr*n_t)
    info = mne.create_info(EEG_NAMES, FS, ["eeg"]*NCH)
    r = mne.io.RawArray(cont, info); r.set_eeg_reference("average", projection=False)
    r.resample(sf); lo, hi = CONFIG["sjepa_bandpass_hz"]; r.filter(lo, hi, method="fir", phase="zero")
    d = r.get_data() * 1e6; nt2 = int(round(n_t*sf/FS)); d = d.reshape(NCH, n_tr, nt2).transpose(1, 0, 2)
    start = int(round(CONFIG["sjepa_window_start_s"]*sf)); start = max(0, min(start, nt2 - W))
    return d[:, :, start:start+W].astype(np.float32)

def build_sjepa(n_times):
    kw = dict(n_chans=NCH, chs_info=None, n_times=n_times, n_outputs=2)
    if CONFIG["sjepa_checkpoint_path"]:
        m = SignalJEPA_PreLocal(**kw)
        m.load_state_dict(torch.load(CONFIG["sjepa_checkpoint_path"], map_location="cpu"), strict=False)
    else:
        m = SignalJEPA_PreLocal.from_pretrained(CONFIG["sjepa_repo_id"], **kw, strict=False)
    for p in m.parameters(): p.requires_grad = False
    return m.to(DEVICE).eval()

@torch.no_grad()
def rich_embeddings(model, X):
    pool = CONFIG["embedding_pool"]; hook = CONFIG["embedding_hook"]
    target = getattr(model, hook); cap = {}
    h = target.register_forward_hook(lambda mod, i, o: cap.__setitem__("z", o.detach()))
    out = []
    try:
        xb = torch.from_numpy(np.asarray(X, dtype=np.float32))
        for i in range(0, len(xb), 32):
            _ = model(xb[i:i+32].to(DEVICE)); z = cap["z"]
            if z.dim() == 2:
                f = z
            else:
                ax = 2 if hook == "spatial_conv" else 1
                f = z.flatten(1) if pool == "flatten" else (z.max(ax).values if pool == "max" else z.mean(ax)).flatten(1)
            out.append(f.cpu().numpy())
    finally:
        h.remove()
    return np.concatenate(out, 0).astype(np.float32)

_SJEPA = [None]
def get_sjepa_features(sid, raw, onsets):
    cache = Path(CONFIG["sjepa_embedding_cache_dir"]) / f"sub-{sid:02d}.npz"
    if cache.exists():
        d = np.load(cache); return d["X"].astype(np.float32)
    if _SJEPA[0] is None: _SJEPA[0] = build_sjepa(CONFIG["sjepa_window_samples"])
    Xw = sjepa_windows(raw, onsets)
    return rich_embeddings(_SJEPA[0], Xw)

def sjepa_proba(E, y, tr, te):
    sc = StandardScaler().fit(E[tr])
    clf = make_lda().fit(sc.transform(E[tr]), y[tr])
    return clf.predict_proba(sc.transform(E[te]))

print("Branch B (frozen S-JEPA embeddings) defined")


[2026-07-07 09:12:25] Branch B (frozen S-JEPA embeddings) defined


# 6. Sanity checks

Explicit, asserted (not assumed) checks run on **every** outer fold inside the CV runner below: (1) train/test trial-index overlap is empty, (2) label balance per split is printed so silent class collapse is visible, (3) each branch + fusion's fold-level prediction is flagged if it collapses onto one class (>95% single-class predictions). For repeated random-holdout CV (used here, matching the rest of this codebase), the **same trial can appear in the test set of more than one repeat** -- that is expected and fine; what must never happen, and is asserted every fold, is a trial appearing in **both** train and test **within the same split**.

In [20]:
def assert_no_overlap(tr, te):
    overlap = set(tr.tolist()) & set(te.tolist())
    assert len(overlap) == 0, f"train/test overlap detected: {overlap}"

def label_balance(y, idx):
    vals, counts = np.unique(y[idx], return_counts=True)
    return dict(zip(vals.tolist(), counts.tolist()))

def collapse_flag(pred):
    c = np.bincount(pred, minlength=2)
    return bool(c.max()/c.sum() > 0.95)

print("Sanity-check helpers defined: assert_no_overlap, label_balance, collapse_flag")


[2026-07-07 09:12:25] Sanity-check helpers defined: assert_no_overlap, label_balance, collapse_flag


# 7. Honest nested-CV stacking runner

Per outer split `(tr, te)`: inner `StratifiedKFold` on `tr` **only** produces out-of-fold probabilities for both branches (`build_oof_meta_features`); a logistic-regression meta-learner is fit on those out-of-fold probabilities. Base learners are then refit on the **full** `tr` and applied once to `te`, together with the meta-learner, for the single honest test-set touch per outer split. Solo-branch predictions (`riemann`, `sjepa`) are also recorded from the same train-only-fit base learners, so fusion-vs-solo is an apples-to-apples comparison on identical splits.

In [21]:
PRED_LOG = {"riemann": {"y_true": [], "y_pred": []},
            "sjepa":   {"y_true": [], "y_pred": []},
            "fusion":  {"y_true": [], "y_pred": []}}

def build_oof_meta_features(seg_by_band, E, y, tr, inner_folds, rng_state):
    skf = StratifiedKFold(n_splits=inner_folds, shuffle=True, random_state=rng_state)
    n = len(tr); oof = np.full((n, 2), np.nan)
    for ia_local, ib_local in skf.split(np.arange(n), y[tr]):
        ia = tr[ia_local]; ib = tr[ib_local]
        assert_no_overlap(ia, ib)
        pa = riemann_multiwindow_proba(seg_by_band, y, ia, ib)[:, 1]
        pb = sjepa_proba(E, y, ia, ib)[:, 1]
        oof[ib_local, 0] = pa; oof[ib_local, 1] = pb
    assert not np.isnan(oof).any(), "inner-CV out-of-fold probabilities left unfilled entries"
    return oof

def run_subject(sid, raw, y, onsets):
    seg_by_band = {bd: band_segment(raw, onsets, bd) for bd in BANDS}
    E = get_sjepa_features(sid, raw, onsets)
    n = len(y); idx = np.arange(n)
    sss = StratifiedShuffleSplit(n_splits=CONFIG["n_repeats"], test_size=CONFIG["test_size"],
                                  random_state=CONFIG["random_state"])
    rows = []
    for fi, (tr, te) in enumerate(sss.split(idx, y)):
        assert_no_overlap(tr, te)
        train_bal, test_bal = label_balance(y, tr), label_balance(y, te)

        oof = build_oof_meta_features(seg_by_band, E, y, tr, CONFIG["inner_folds"], CONFIG["random_state"] + fi)
        meta = LogisticRegression(C=CONFIG["meta_C"]).fit(oof, y[tr])

        probaA_te = riemann_multiwindow_proba(seg_by_band, y, tr, te)   # base learners refit on full tr
        probaB_te = sjepa_proba(E, y, tr, te)
        meta_in_te = np.column_stack([probaA_te[:, 1], probaB_te[:, 1]])

        pred_riemann = np.argmax(probaA_te, axis=1)
        pred_sjepa = np.argmax(probaB_te, axis=1)
        pred_fusion = meta.predict(meta_in_te)

        for br, pred in (("riemann", pred_riemann), ("sjepa", pred_sjepa), ("fusion", pred_fusion)):
            PRED_LOG[br]["y_true"].extend(y[te].tolist()); PRED_LOG[br]["y_pred"].extend(pred.tolist())

        row = {"subject": sid, "fold": fi,
               "train_n": len(tr), "test_n": len(te),
               "train_balance": json.dumps(train_bal), "test_balance": json.dumps(test_bal),
               "overlap_ok": True}   # assert_no_overlap above would have raised otherwise
        for br, pred in (("riemann", pred_riemann), ("sjepa", pred_sjepa), ("fusion", pred_fusion)):
            row[f"{br}_acc"] = accuracy_score(y[te], pred)
            row[f"{br}_bacc"] = balanced_accuracy_score(y[te], pred)
            row[f"{br}_collapse"] = collapse_flag(pred)
        rows.append(row)
    return rows

def run_all():
    files = find_files(CONFIG["source_roots"]); keep = CONFIG["subjects_to_use"]; out = []; t0 = time.time()
    for path in files:
        sid = int(os.path.basename(path).split("-")[1][:2])
        if keep is not None and sid not in keep: continue
        raw, y, onsets = load_raw(path)
        rows = run_subject(sid, raw, y, onsets); out.extend(rows)
        msg = " ".join(f"{br}={np.mean([r[f'{br}_bacc'] for r in rows])*100:4.1f}" for br in ["riemann", "sjepa", "fusion"])
        print(f"sub-{sid:02d}: {msg}  ({time.time()-t0:5.1f}s)")
    return pd.DataFrame(out)

RESULTS = run_all()
RESULTS.head()


[2026-07-07 09:12:25] source: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_data/liu2024_figshare/sourcedata (50 subjects)
[2026-07-07 09:12:27] sub-01: riemann=68.8 sjepa=62.5 fusion=37.5  (  2.1s)
[2026-07-07 09:12:28] sub-02: riemann=56.2 sjepa=68.8 fusion=68.8  (  3.3s)
[2026-07-07 09:12:29] sub-03: riemann=37.5 sjepa=62.5 fusion=37.5  (  4.6s)
[2026-07-07 09:12:31] sub-04: riemann=62.5 sjepa=56.2 fusion=56.2  (  5.8s)
[2026-07-07 09:12:32] sub-05: riemann=50.0 sjepa=62.5 fusion=62.5  (  7.0s)
[2026-07-07 09:12:33] sub-06: riemann=37.5 sjepa=56.2 fusion=50.0  (  8.4s)
[2026-07-07 09:12:34] sub-07: riemann=62.5 sjepa=75.0 fusion=75.0  (  9.6s)
[2026-07-07 09:12:36] sub-08: riemann=62.5 sjepa=37.5 fusion=43.8  ( 10.8s)
[2026-07-07 09:12:37] sub-09: riemann=50.0 sjepa=56.2 fusion=43.8  ( 12.6s)
[2026-07-07 09:12:39] sub-10: riemann=56.2 sjepa=62.5 fusion=62.5  ( 13.8s)
[2026-07-07 09:12:40] sub-11: riemann=62.5 sjepa=56.2 fusion=37.5  ( 15.1s

,subject,fold,train_n,test_n,train_balance,test_balance,overlap_ok,riemann_acc,riemann_bacc,riemann_collapse,sjepa_acc,sjepa_bacc,sjepa_collapse,fusion_acc,fusion_bacc,fusion_collapse
0,1,0,24,16,"{""0"": 12, ""1"": 12}","{""0"": 8, ""1"": 8}",True,0.6875,0.6875,False,0.6250,0.6250,False,0.3750,0.3750,False
1,2,0,24,16,"{""0"": 12, ""1"": 12}","{""0"": 8, ""1"": 8}",True,0.5625,0.5625,False,0.6875,0.6875,False,0.6875,0.6875,False
2,3,0,24,16,"{""0"": 12, ""1"": 12}","{""0"": 8, ""1"": 8}",True,0.3750,0.3750,False,0.6250,0.6250,False,0.3750,0.3750,False
3,4,0,24,16,"{""0"": 12, ""1"": 12}","{""0"": 8, ""1"": 8}",True,0.6250,0.6250,False,0.5625,0.5625,False,0.5625,0.5625,False
4,5,0,24,16,"{""0"": 12, ""1"": 12}","{""0"": 8, ""1"": 8}",True,0.5000,0.5000,False,0.6250,0.6250,False,0.6250,0.6250,False


# 8. Sanity-check verification

Aggregate confirmation that every fold in `RESULTS` passed the explicit checks from section 6 -- printed, not just assumed.

In [22]:
assert RESULTS["overlap_ok"].all(), "some fold recorded a train/test overlap"
print(f"Train/test overlap check: PASS on all {len(RESULTS)} folds across {RESULTS['subject'].nunique()} subject(s)")

for _, r in RESULTS.iterrows():
    tb, teb = json.loads(r["train_balance"]), json.loads(r["test_balance"])
    print(f"  sub-{int(r['subject']):02d} fold{int(r['fold'])}: train n={r['train_n']} balance={tb}  "
          f"test n={r['test_n']} balance={teb}")

collapse_rates = {br: RESULTS[f"{br}_collapse"].mean() for br in ["riemann", "sjepa", "fusion"]}
print(f"Collapse rate (fold predicts >95% one class): {collapse_rates}")


[2026-07-07 09:13:25] Train/test overlap check: PASS on all 50 folds across 50 subject(s)
[2026-07-07 09:13:25]   sub-01 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-02 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-03 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-04 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-05 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-06 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-07 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09:13:25]   sub-08 fold0: train n=24 balance={'0': 12, '1': 12}  test n=16 balance={'0': 8, '1': 8}
[2026-07-07 09

# 9. Results -- per-subject accuracy, mean, SD, Wilcoxon

In [23]:
def summarize(df):
    print("="*70)
    print(f"Multi-window Riemannian ensemble x frozen S-JEPA stacking -- Liu2024, n={df['subject'].nunique()} subject(s)")
    subj = df.groupby("subject").mean(numeric_only=True)
    for br in ["riemann", "sjepa", "fusion"]:
        print(f"  {br:8s}: bal-acc {subj[f'{br}_bacc'].mean()*100:5.2f}% +/- {subj[f'{br}_bacc'].std(ddof=0)*100:4.2f}  "
              f"acc {subj[f'{br}_acc'].mean()*100:5.2f}%  collapse {df[f'{br}_collapse'].mean()*100:3.0f}%")
    if len(subj) > 1:
        try:
            w, p = wilcoxon(subj["fusion_bacc"], subj["riemann_bacc"])
            print(f"  Wilcoxon fusion vs riemann: dMean={(subj['fusion_bacc']-subj['riemann_bacc']).mean()*100:+.2f}pp p={p:.4f}")
        except Exception as e:
            print("  wilcoxon fusion-vs-riemann n/a:", e)
        try:
            w, p = wilcoxon(subj["fusion_bacc"], subj["sjepa_bacc"])
            print(f"  Wilcoxon fusion vs sjepa  : dMean={(subj['fusion_bacc']-subj['sjepa_bacc']).mean()*100:+.2f}pp p={p:.4f}")
        except Exception as e:
            print("  wilcoxon fusion-vs-sjepa n/a:", e)
    else:
        print("  (Wilcoxon needs >1 subject; this historical smoke must not be scaled as-is.)")
    print("="*70)
    return subj

SUBJ = summarize(RESULTS)
SUBJ[[c for c in SUBJ.columns if c.endswith("_bacc") or c.endswith("_acc")]]


[2026-07-07 09:13:25] ======================================================================
[2026-07-07 09:13:25] Multi-window Riemannian ensemble x frozen S-JEPA stacking -- Liu2024, n=50 subject(s)
[2026-07-07 09:13:25]   riemann : bal-acc 52.12% +/- 13.56  acc 52.12%  collapse   0%
[2026-07-07 09:13:25]   sjepa   : bal-acc 54.25% +/- 9.39  acc 54.25%  collapse   0%
[2026-07-07 09:13:25]   fusion  : bal-acc 50.25% +/- 11.45  acc 50.25%  collapse   2%
[2026-07-07 09:13:25]   Wilcoxon fusion vs riemann: dMean=-1.88pp p=0.5094
[2026-07-07 09:13:25]   Wilcoxon fusion vs sjepa  : dMean=-4.00pp p=0.0556
[2026-07-07 09:13:25] ======================================================================


,riemann_acc,riemann_bacc,sjepa_acc,sjepa_bacc,fusion_acc,fusion_bacc
subject,,,,,,
1,0.6875,0.6875,0.6250,0.6250,0.3750,0.3750
2,0.5625,0.5625,0.6875,0.6875,0.6875,0.6875
3,0.3750,0.3750,0.6250,0.6250,0.3750,0.3750
4,0.6250,0.6250,0.5625,0.5625,0.5625,0.5625
5,0.5000,0.5000,0.6250,0.6250,0.6250,0.6250
6,0.3750,0.3750,0.5625,0.5625,0.5000,0.5000
7,0.6250,0.6250,0.7500,0.7500,0.7500,0.7500
8,0.6250,0.6250,0.3750,0.3750,0.4375,0.4375
9,0.5000,0.5000,0.5625,0.5625,0.4375,0.4375


# 10. Confusion matrices (pooled across folds/subjects)

In [24]:
CONF_MATS = {}
for br in ["riemann", "sjepa", "fusion"]:
    yt, yp = PRED_LOG[br]["y_true"], PRED_LOG[br]["y_pred"]
    CONF_MATS[br] = confusion_matrix(yt, yp, labels=[0, 1])
    print(f"{br} confusion matrix (rows=true[left,right], cols=pred[left,right]):\n{CONF_MATS[br]}\n")

if HAVE_MPL:
    fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
    for ax, br in zip(axes, ["riemann", "sjepa", "fusion"]):
        im = ax.imshow(CONF_MATS[br], cmap="Blues")
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(CONF_MATS[br][i, j]), ha="center", va="center")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["left", "right"])
        ax.set_yticks([0, 1]); ax.set_yticklabels(["left", "right"])
        ax.set_title(br); ax.set_xlabel("predicted"); ax.set_ylabel("true")
    plt.tight_layout()


[2026-07-07 09:13:25] riemann confusion matrix (rows=true[left,right], cols=pred[left,right]):
[[234 166]
 [217 183]]

[2026-07-07 09:13:25] sjepa confusion matrix (rows=true[left,right], cols=pred[left,right]):
[[223 177]
 [189 211]]

[2026-07-07 09:13:25] fusion confusion matrix (rows=true[left,right], cols=pred[left,right]):
[[191 209]
 [189 211]]



# 11. Comparison against existing baselines

**Read the 'fair?' column before comparing rows.** Baseline numbers were not all produced with the same subject set or CV scheme as this notebook. Do not rerun this stopped route merely to force an apples-to-apples comparison; a new study requires a materially new prespecified hypothesis.

In [25]:
baseline_rows = [
    {"method": "CSP+LDA (honest)", "subjects": "all 50", "cv_scheme": "not verified here", "bal_acc_pct": 50.58,
     "fair_vs_this_run": "No -- different pipeline/CV scheme; provenance not re-verified in this notebook."},
    {"method": "FBCSP+SVM (honest)", "subjects": "all 50", "cv_scheme": "not verified here", "bal_acc_pct": 51.15,
     "fair_vs_this_run": "No -- different pipeline/CV scheme; provenance not re-verified in this notebook."},
    {"method": "TWFB+DGFMDM paper-faithful, honest nested-CV", "subjects": "all 50",
     "cv_scheme": "repeated_holdout 24/16 x5, inner_folds=3", "bal_acc_pct": 51.60,
     "fair_vs_this_run": "Partial -- same honest-nested-CV *philosophy*, different feature/classifier "
                         "and outer test_size/n_repeats than this notebook's CONFIG."},
    {"method": "TWFB+DGFMDM paper-faithful, honest nested-CV", "subjects": "Lv-14 subset",
     "cv_scheme": "repeated_holdout 24/16 x5, inner_folds=3", "bal_acc_pct": 50.80,
     "fair_vs_this_run": "Partial -- same subject-set-selection philosophy as this notebook's Lv-14 "
                         "runs, but different CV scheme/test_size."},
    {"method": "Frozen S-JEPA embeddings + LDA probe (solo)", "subjects": "all 50 (reported)",
     "cv_scheme": "sjepa_5fold or liu_repeated_holdout -- exact protocol match to this run NOT verified "
                  "(open item, CLAUDE.md sec.3 #4)", "bal_acc_pct": 53.95,
     "fair_vs_this_run": "No -- exact CV protocol match unconfirmed; treat as directional only until re-run."},
]
BASELINES = pd.DataFrame(baseline_rows)

this_run_rows = [
    {"method": f"{br} (this run)",
     "subjects": f"{RESULTS['subject'].nunique()} subject(s): {sorted(RESULTS['subject'].unique().tolist())}",
     "cv_scheme": f"repeated_holdout test_size={CONFIG['test_size']} x{CONFIG['n_repeats']}, inner_folds={CONFIG['inner_folds']}",
     "bal_acc_pct": round(SUBJ[f"{br}_bacc"].mean()*100, 2),
     "fair_vs_this_run": "Yes -- identical subject set, splits, and CV scheme as the other 'this run' rows."}
    for br in ["riemann", "sjepa", "fusion"]
]
THIS_RUN = pd.DataFrame(this_run_rows)

COMPARISON = pd.concat([THIS_RUN, BASELINES], ignore_index=True)
COMPARISON


,method,subjects,cv_scheme,bal_acc_pct,fair_vs_this_run
0,riemann (this run),"50 subject(s): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,...","repeated_holdout test_size=0.4 x1, inner_folds=2",52.12,"Yes -- identical subject set, splits, and CV s..."
1,sjepa (this run),"50 subject(s): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,...","repeated_holdout test_size=0.4 x1, inner_folds=2",54.25,"Yes -- identical subject set, splits, and CV s..."
2,fusion (this run),"50 subject(s): [1, 2, 3, 4, 5, 6, 7, 8, 9, 10,...","repeated_holdout test_size=0.4 x1, inner_folds=2",50.25,"Yes -- identical subject set, splits, and CV s..."
3,CSP+LDA (honest),all 50,not verified here,50.58,No -- different pipeline/CV scheme; provenance...
4,FBCSP+SVM (honest),all 50,not verified here,51.15,No -- different pipeline/CV scheme; provenance...
5,"TWFB+DGFMDM paper-faithful, honest nested-CV",all 50,"repeated_holdout 24/16 x5, inner_folds=3",51.60,"Partial -- same honest-nested-CV *philosophy*,..."
6,"TWFB+DGFMDM paper-faithful, honest nested-CV",Lv-14 subset,"repeated_holdout 24/16 x5, inner_folds=3",50.80,Partial -- same subject-set-selection philosop...
7,Frozen S-JEPA embeddings + LDA probe (solo),all 50 (reported),sjepa_5fold or liu_repeated_holdout -- exact p...,53.95,No -- exact CV protocol match unconfirmed; tre...


# 12. Save artifacts

In [26]:
RESULTS.to_csv(ARTIFACT_DIR/"fold_results.csv", index=False)
SUBJ.to_csv(ARTIFACT_DIR/"subject_summary.csv")
COMPARISON.to_csv(ARTIFACT_DIR/"comparison_vs_baselines.csv", index=False)
for br in ["riemann", "sjepa", "fusion"]:
    np.savetxt(ARTIFACT_DIR/f"confusion_matrix_{br}.csv", CONF_MATS[br], fmt="%d", delimiter=",")

gsum = {br: {"bacc_mean": float(SUBJ[f"{br}_bacc"].mean()),
             "bacc_std": float(SUBJ[f"{br}_bacc"].std(ddof=0)),
             "collapse_rate": float(RESULTS[f"{br}_collapse"].mean())}
        for br in ["riemann", "sjepa", "fusion"]}
json.dump({"config": {k: (str(v) if isinstance(v, Path) else v) for k, v in CONFIG.items()},
           "summary": gsum, "n_subjects": int(RESULTS["subject"].nunique()),
           "overlap_check_passed": bool(RESULTS["overlap_ok"].all())},
          open(ARTIFACT_DIR/"global_metrics.json", "w"), indent=2, default=str)
json.dump(RESULTS.to_dict(orient="records"), open(ARTIFACT_DIR/"subject_metrics.json", "w"), indent=2, default=str)

if HAVE_MPL:
    fig, ax = plt.subplots(figsize=(6, 4))
    brs = ["riemann", "sjepa", "fusion"]
    m = [SUBJ[f"{b}_bacc"].mean()*100 for b in brs]; e = [SUBJ[f"{b}_bacc"].std(ddof=0)*100 for b in brs]
    ax.bar(brs, m, yerr=e, capsize=5, color=["#88c", "#4a4", "#c44"])
    ax.axhline(50, ls=":", c="gray", label="chance")
    ax.set_ylabel("balanced accuracy (%)"); ax.set_title(f"Multi-window Riemann x S-JEPA stacking (n={RESULTS['subject'].nunique()})")
    ax.legend(fontsize=8)
    for i, v in enumerate(m): ax.text(i, v+1, f"{v:.1f}", ha="center")
    plt.tight_layout(); plt.savefig(ARTIFACT_DIR/"branch_balacc.png", dpi=150, bbox_inches="tight"); plt.close(fig)

print("Saved: fold_results.csv, subject_summary.csv, comparison_vs_baselines.csv, confusion_matrix_*.csv,")
print("       global_metrics.json, subject_metrics.json, branch_balacc.png")
print(f"Artifacts: {ARTIFACT_DIR}")


[2026-07-07 09:13:25] Saved: fold_results.csv, subject_summary.csv, comparison_vs_baselines.csv, confusion_matrix_*.csv,
[2026-07-07 09:13:25]        global_metrics.json, subject_metrics.json, branch_balacc.png
[2026-07-07 09:13:25] Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-multiwindow-riemann-sjepa-stacking/20260707_0912_700a2c85


# 13. Scaling status

The former scale-up instructions and deleted sweep config are intentionally not retained as
runnable directions. The completed prototype and later fusion nulls close this exact route. A
future run requires a materially new hypothesis, prespecified protocol, and new config name.


# 14. Notes

- **Leakage:** every fitted step (per-window TangentSpace + LDA in Branch A, LDA in Branch B, and
  the logistic meta-learner) is fit on train-only indices; Branch A's window ensemble is a fixed
  average, not a test-visible selection; Branch B's embeddings are frozen/label-free. Sections 6-8
  assert this explicitly per fold rather than assuming it.
- **What would count as a fair improvement:** the fusion branch's honest mean per-subject balanced
  accuracy should (a) beat the frozen-S-JEPA-solo probe's ~53.95% *on the same subject set and CV
  scheme* -- not against the numbers in Section 11's baseline table, which use different protocols
  -- and (b) show a significant paired per-subject Wilcoxon improvement over **both** solo branches,
  not just a higher mean with overlapping spread across only a couple of subjects.
- **No follow-up ablations are authorized from this notebook's outer results.** The exact route is
  closed in `AGENTS.md` section 2e; any future fusion study needs a new hypothesis and protocol.
- **If fusion ~= max(riemann, sjepa) with Wilcoxon n.s.:** that is a credible, defensible result
  ("stacking doesn't help beyond the better solo branch here"), not a failure -- report it as such,
  consistent with this project's existing honest-evaluation framing (CLAUDE.md sec.2d).
